#1. Train the 3D Gaussian Splatting model

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

In [ ]:
import tensorflow as tf

print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))
print("TensorFlow version: ", tf.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
FOLDERNAME = 'cs231n/project/'
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))
# !cp -r . /content/drive/MyDrive/gaussian-splatting/

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
%cd /content
!git clone --recursive https://github.com/camenduru/gaussian-splatting

!pip install -q plyfile

%cd /content/gaussian-splatting
!pip install -q /content/gaussian-splatting/submodules/diff-gaussian-rasterization
!pip install -q /content/gaussian-splatting/submodules/simple-knn

!wget https://huggingface.co/camenduru/gaussian-splatting/resolve/main/tandt_db.zip
!yes A | unzip tandt_db.zip  # Automatically answer "All" to the unzip prompts

# Modify the train.py script to move model and data to GPU
!sed -i 's/device = torch.device("cpu")/device = torch.device("cuda" if torch.cuda.is_available() else "cpu")/g' train.py
!sed -i 's/model = model.to(device)/model = model.to(device)/g' train.py
!sed -i 's/data, target = data.to(device), target.to(device)/data, target = data.to(device), target.to(device)/g' train.py

!python train.py -s /content/gaussian-splatting/tandt/train

# !wget https://huggingface.co/camenduru/gaussian-splatting/resolve/main/GaussianViewTest.zip
# !unzip GaussianViewTest.zip
# !python render.py -m /content/gaussian-splatting/GaussianViewTest/model
# !ffmpeg -framerate 3 -i /content/gaussian-splatting/GaussianViewTest/model/train/ours_30000/renders/%05d.png -vf "pad=ceil(iw/2)*2:ceil(ih/2)*2" -c:v libx264 -r 3 -pix_fmt yuv420p /content/renders.mp4
# !ffmpeg -framerate 3 -i /content/gaussian-splatting/GaussianViewTest/model/train/ours_30000/gt/%05d.png -vf "pad=ceil(iw/2)*2:ceil(ih/2)*2" -c:v libx264 -r 3 -pix_fmt yuv420p /content/gt.mp4 -y

#2. Render the trained model

In [ ]:
# Define the paths and parameters
output_model_dir = '/content/gaussian-splatting/output/d62aca2f-e'

# Modify the render.py script to use GPU
!sed -i 's/device = torch.device("cpu")/device = torch.device("cuda" if torch.cuda.is_available() else "cpu")/g' render.py
!sed -i 's/model = model.to(device)/model = model.to(device)/g' render.py

# Run the rendering script
!python render.py --model_path {output_model_dir} --iteration 30000

In [ ]:
!ls /content/gaussian-splatting/output/d62aca2f-e/train/ours_30000/renders
!ls /content/gaussian-splatting/output/d62aca2f-e/test/ours_30000/renders

In [ ]:
!ffmpeg -y -framerate 3 -i /content/gaussian-splatting/output/d62aca2f-e/train/ours_30000/renders/%05d.png -vf "pad=ceil(iw/2)*2:ceil(ih/2)*2" -c:v libx264 -r 3 -pix_fmt yuv420p /content/train_renders.mp4

In [ ]:
!ls /content/train_renders.mp4

In [ ]:
from google.colab import files
files.download('/content/train_renders.mp4')

#3. Set up the viewer

In [ ]:
!git clone https://github.com/camenduru/splat
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /content/cloudflared-linux-amd64.deb
!dpkg -i /content/cloudflared-linux-amd64.deb

In [ ]:
%cd /content
!git clone https://github.com/camenduru/splat

In [ ]:
!pip install pyngrok
from pyngrok import ngrok

# Replace "YOUR_AUTHTOKEN" with the authtoken you got from ngrok
ngrok.set_auth_token("2gWHu0eX53vyboV6uhJrDSIAGWR_6dcTE3DgBLkm3ZLz73svy")

In [ ]:
from pyngrok import ngrok

# Start an HTTP tunnel on the default port 7860
http_tunnel = ngrok.connect(7860, "http")
print(f"Ngrok tunnel available at: {http_tunnel.public_url}")

# Change directory to the splat repository
%cd /content/splat

# Start the HTTP server
!python -m http.server 7860

#4. Evaluate

In [ ]:
!ls /content/gaussian-splatting/output/38d71606-8
test_data_path = '/content/gaussian-splatting/test_data'
!ls /content/gaussian-splatting/test_data

In [ ]:
output_model_dir = '/content/gaussian-splatting/output/38d71606-8'
cfgfilepath = f'{output_model_dir}/cfg_args'

with open(cfgfilepath, 'w') as cfg_file:
    cfg_file.write('--dataset_path /path/to/dataset\n--network_depth 8\n--num_gaussians 1024\n--learning_rate 0.001\n--batch_size 4')

In [ ]:
output_model_dir = '/content/gaussian-splatting/output/38d71606-8'
test_data_path = '/path/to/test/data'  # Replace with the actual path to your test data

!python /content/gaussian-splatting/render.py --model_path {output_model_dir} --iteration 30000 --eval --source_path {test_data_path}